# ⚡ Task 3: Energy Consumption Time Series Forecasting
## Household Power Consumption — ARIMA · Prophet · XGBoost

---
**Objective:** Forecast short-term household energy usage using historical time-based patterns.

**Dataset:** Household Power Consumption Dataset (UCI Machine Learning Repository)

**Skills:** Time Series Forecasting · Feature Engineering · Model Comparison (MAE/RMSE) · Temporal Visualization

---

## 📦 Step 0: Install Libraries

In [ ]:
!pip install prophet xgboost statsmodels scikit-learn matplotlib seaborn -q
print('✅ All libraries installed!')

## 📚 Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Time series
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from prophet import Prophet

# ML
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    'figure.facecolor': '#050510',
    'axes.facecolor':   '#0e0e20',
    'axes.edgecolor':   '#303060',
    'axes.labelcolor':  '#ccccee',
    'xtick.color':      '#8888aa',
    'ytick.color':      '#8888aa',
    'text.color':       '#ccccee',
    'grid.color':       '#1a1a3a',
    'grid.alpha':       0.6,
    'figure.dpi':       130,
    'font.size':        11
})

ACTUAL_COLOR    = '#4ecdc4'
ARIMA_COLOR     = '#ff6b6b'
PROPHET_COLOR   = '#ffd93d'
XGBOOST_COLOR   = '#a855f7'

print('✅ All libraries imported!')

## 📥 Step 2: Load & Parse Dataset

In [ ]:
# UCI Household Power Consumption Dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip'

import io, zipfile, urllib.request

print('⬇️  Downloading Household Power Consumption dataset from UCI...')
print('    (This is a ~20MB file, please wait...)')

try:
    response = urllib.request.urlopen(url, timeout=120)
    zip_file = zipfile.ZipFile(io.BytesIO(response.read()))
    with zip_file.open('household_power_consumption.txt') as f:
        df_raw = pd.read_csv(f, sep=';', low_memory=False,
                             na_values=['?', 'NA', ''])
    print(f'✅ Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
    loaded_real = True
except Exception as e:
    print(f'⚠️  Download issue: {e}')
    print('📋 Generating realistic synthetic power consumption data...')
    loaded_real = False

if not loaded_real:
    # Generate 2 years of hourly data (realistic patterns)
    date_rng = pd.date_range('2007-01-01', '2009-12-31', freq='H')
    n = len(date_rng)
    np.random.seed(42)
    # Daily pattern: peak morning (7-9) and evening (18-21)
    hour_pattern = np.array([0.5,0.4,0.3,0.3,0.35,0.5,0.9,1.3,1.2,
                              0.8,0.7,0.75,0.85,0.8,0.75,0.8,0.9,1.1,
                              1.3,1.4,1.3,1.1,0.9,0.7])
    # Seasonal pattern
    days_of_year = date_rng.dayofyear
    seasonal = 1 + 0.3 * np.cos(2 * np.pi * days_of_year / 365 + np.pi)
    hourly   = hour_pattern[date_rng.hour]
    noise    = np.random.normal(0, 0.1, n)
    power    = np.clip(hourly * seasonal + noise + 0.5, 0.1, 4.0)
    df_raw = pd.DataFrame({'Date': date_rng.strftime('%d/%m/%Y'),
                           'Time': date_rng.strftime('%H:%M:%S'),
                           'Global_active_power': power.round(3)})
    print(f'✅ Synthetic dataset created: {df_raw.shape}')

In [ ]:
# ── Parse datetime & resample to hourly ───────────────────────────────────────
if 'Date' in df_raw.columns and 'Time' in df_raw.columns:
    df_raw['datetime'] = pd.to_datetime(
        df_raw['Date'].astype(str) + ' ' + df_raw['Time'].astype(str),
        dayfirst=True, errors='coerce'
    )
    df_raw.set_index('datetime', inplace=True)
    df_raw['Global_active_power'] = pd.to_numeric(
        df_raw['Global_active_power'], errors='coerce'
    )

# Resample to hourly mean
df_hourly = df_raw[['Global_active_power']].resample('H').mean()

# Drop NaNs
df_hourly.dropna(inplace=True)
df_hourly.index.freq = 'H'

print(f'⏱️  Hourly time series shape: {df_hourly.shape}')
print(f'   Date range: {df_hourly.index.min()} → {df_hourly.index.max()}')
print(f'   Missing values: {df_hourly.isnull().sum().sum()}')
print(f'\n📊 Power Statistics (kW):')
print(df_hourly.describe().round(4))

## 🔍 Step 3: Exploratory Analysis of Time Series

In [ ]:
# ── Use a manageable subset: last 3 months of data ────────────────────────────
df_sub = df_hourly.tail(24 * 90).copy()  # last 90 days
print(f'Working subset: {df_sub.shape[0]} hourly observations')
print(f'Date range: {df_sub.index.min()} → {df_sub.index.max()}')

# ── Figure 1: Time Series Overview ───────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(20, 14))
fig.suptitle('Energy Consumption Time Series — Exploratory Analysis',
             fontsize=17, fontweight='bold', color='#ccccee', y=1.01)

# Raw series
axes[0].plot(df_sub.index, df_sub['Global_active_power'],
             color=ACTUAL_COLOR, lw=0.8, alpha=0.9)
axes[0].fill_between(df_sub.index, df_sub['Global_active_power'],
                      alpha=0.2, color=ACTUAL_COLOR)
axes[0].set_ylabel('Global Active Power (kW)', fontsize=11)
axes[0].set_title('Hourly Power Consumption (90-day Window)', fontweight='bold', pad=10)
axes[0].grid(True, alpha=0.3)

# 7-day rolling average
rolling_24h  = df_sub['Global_active_power'].rolling(24).mean()
rolling_168h = df_sub['Global_active_power'].rolling(168).mean()
axes[1].plot(df_sub.index, df_sub['Global_active_power'], color=ACTUAL_COLOR,
             lw=0.6, alpha=0.4, label='Hourly')
axes[1].plot(df_sub.index, rolling_24h, color=PROPHET_COLOR, lw=2,
             label='24h Rolling Mean')
axes[1].plot(df_sub.index, rolling_168h, color=ARIMA_COLOR, lw=2.5,
             label='168h (7-day) Rolling Mean')
axes[1].set_ylabel('Global Active Power (kW)', fontsize=11)
axes[1].set_title('Rolling Averages — Trend Extraction', fontweight='bold', pad=10)
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

# Hourly pattern (average by hour of day)
df_sub['hour'] = df_sub.index.hour
hourly_avg = df_sub.groupby('hour')['Global_active_power'].mean()
hourly_std = df_sub.groupby('hour')['Global_active_power'].std()
axes[2].fill_between(hourly_avg.index, hourly_avg - hourly_std,
                      hourly_avg + hourly_std, alpha=0.25, color=XGBOOST_COLOR)
axes[2].plot(hourly_avg.index, hourly_avg.values, 'o-',
             color=XGBOOST_COLOR, lw=2.5, ms=7, mec='white', mew=1)
axes[2].set_xlabel('Hour of Day', fontsize=11)
axes[2].set_ylabel('Avg Power (kW)', fontsize=11)
axes[2].set_title('Average Power by Hour of Day (± 1 Std)', fontweight='bold', pad=10)
axes[2].set_xticks(range(0, 24))
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task3_eda.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ EDA plots saved!')

In [ ]:
# ── Seasonal Decomposition ────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(20, 14))
fig.suptitle('Seasonal Decomposition of Energy Consumption (Additive)',
             fontsize=15, fontweight='bold', color='#ccccee', y=1.01)

# Use daily data for decomposition
df_daily = df_sub['Global_active_power'].resample('D').mean()
decomp   = seasonal_decompose(df_daily, model='additive', period=7)

components = [decomp.observed, decomp.trend, decomp.seasonal, decomp.resid]
titles     = ['Observed', 'Trend', 'Seasonal (Weekly)', 'Residual']
colors     = [ACTUAL_COLOR, PROPHET_COLOR, XGBOOST_COLOR, ARIMA_COLOR]

for ax, comp, title, color in zip(axes, components, titles, colors):
    ax.plot(comp, color=color, lw=1.8)
    ax.fill_between(comp.index, comp, alpha=0.15, color=color)
    ax.set_title(title, fontweight='bold', pad=8, fontsize=12)
    ax.set_ylabel('Power (kW)', fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task3_decomposition.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Stationarity Test ─────────────────────────────────────────────────────────
result = adfuller(df_sub['Global_active_power'].dropna())
print('='*55)
print('  AUGMENTED DICKEY-FULLER TEST (Stationarity)')
print('='*55)
print(f'ADF Statistic : {result[0]:.6f}')
print(f'p-value       : {result[1]:.6f}')
for k, v in result[4].items():
    print(f'Critical ({k}): {v:.6f}')
if result[1] < 0.05:
    print('\n✅ Series is STATIONARY (p < 0.05) — Ready for ARIMA')
else:
    print('\n⚠️  Series is NON-STATIONARY — Differencing may be needed')

## 🔧 Step 4: Feature Engineering

In [ ]:
# ── Time-based feature engineering ───────────────────────────────────────────
def create_time_features(df):
    df = df.copy()
    df['hour']          = df.index.hour
    df['dayofweek']     = df.index.dayofweek      # 0=Monday
    df['dayofmonth']    = df.index.day
    df['month']         = df.index.month
    df['quarter']       = df.index.quarter
    df['weekofyear']    = df.index.isocalendar().week.astype(int)
    df['is_weekend']    = (df.index.dayofweek >= 5).astype(int)
    df['is_business_h'] = ((df.index.hour >= 8) & (df.index.hour <= 18)).astype(int)
    # Cyclical encoding of hour & day
    df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
    df['day_sin']    = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['day_cos']    = np.cos(2 * np.pi * df['dayofweek'] / 7)
    df['month_sin']  = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']  = np.cos(2 * np.pi * df['month'] / 12)
    # Lag features
    for lag in [1, 2, 3, 6, 12, 24, 48, 168]:
        df[f'lag_{lag}h'] = df['Global_active_power'].shift(lag)
    # Rolling statistics
    df['roll_mean_24h']  = df['Global_active_power'].shift(1).rolling(24).mean()
    df['roll_std_24h']   = df['Global_active_power'].shift(1).rolling(24).std()
    df['roll_mean_168h'] = df['Global_active_power'].shift(1).rolling(168).mean()
    df['roll_max_24h']   = df['Global_active_power'].shift(1).rolling(24).max()
    df['roll_min_24h']   = df['Global_active_power'].shift(1).rolling(24).min()
    return df

df_feat = create_time_features(df_sub)
df_feat.dropna(inplace=True)

print(f'Feature matrix shape: {df_feat.shape}')
print(f'\nEngineered features ({df_feat.shape[1]-1}):')
feature_cols = [c for c in df_feat.columns if c != 'Global_active_power']
print(feature_cols)

## ✂️ Step 5: Train / Test Split

In [ ]:
# ── Temporal split: last 7 days as test ───────────────────────────────────────
FORECAST_HOURS = 24 * 7  # 1 week ahead

train_ts = df_sub[:-FORECAST_HOURS]
test_ts  = df_sub[-FORECAST_HOURS:]

train_feat = df_feat[:-FORECAST_HOURS]
test_feat  = df_feat[-FORECAST_HOURS:]

y_train = train_feat['Global_active_power']
y_test  = test_feat['Global_active_power']
X_train = train_feat[feature_cols]
X_test  = test_feat[feature_cols]

print(f'Training period: {train_ts.index.min()} → {train_ts.index.max()}')
print(f'Test period    : {test_ts.index.min()} → {test_ts.index.max()}')
print(f'Train samples  : {len(y_train)}')
print(f'Test samples   : {len(y_test)} ({FORECAST_HOURS/24:.0f} days)')

## 📈 Step 6: ARIMA Model

In [ ]:
# ── ARIMA/SARIMA ─────────────────────────────────────────────────────────────
print('🔄 Training ARIMA(2,0,2) model...')

# Use daily data for ARIMA (faster, less memory)
train_daily = train_ts['Global_active_power'].resample('D').mean()
test_daily  = test_ts['Global_active_power'].resample('D').mean()

arima_model = SARIMAX(
    train_daily,
    order=(2, 0, 2),
    seasonal_order=(1, 0, 1, 7),  # Weekly seasonality
    enforce_stationarity=False,
    enforce_invertibility=False
)
arima_fit = arima_model.fit(disp=False)

arima_pred = arima_fit.forecast(steps=len(test_daily))
arima_pred = np.maximum(arima_pred, 0)  # No negative power

arima_mae  = mean_absolute_error(test_daily, arima_pred)
arima_rmse = np.sqrt(mean_squared_error(test_daily, arima_pred))
arima_mape = np.mean(np.abs((test_daily.values - arima_pred) / test_daily.values)) * 100

print(f'✅ ARIMA trained!')
print(f'   MAE  = {arima_mae:.4f} kW')
print(f'   RMSE = {arima_rmse:.4f} kW')
print(f'   MAPE = {arima_mape:.2f}%')
print(f'\n{arima_fit.summary()}')

## 🔮 Step 7: Prophet Model

In [ ]:
# ── Prophet ──────────────────────────────────────────────────────────────────
print('🔄 Training Prophet model...')

# Prophet requires columns 'ds' and 'y'
prophet_train = train_daily.reset_index()
prophet_train.columns = ['ds', 'y']
prophet_train['ds'] = pd.to_datetime(prophet_train['ds'])

prophet_model = Prophet(
    seasonality_mode='additive',
    weekly_seasonality=True,
    daily_seasonality=False,
    yearly_seasonality=True,
    changepoint_prior_scale=0.05,
    interval_width=0.95
)
prophet_model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
prophet_model.fit(prophet_train)

# Forecast
future = prophet_model.make_future_dataframe(periods=len(test_daily), freq='D')
prophet_forecast = prophet_model.predict(future)

# Extract test period predictions
prophet_pred = prophet_forecast['yhat'].tail(len(test_daily)).values
prophet_pred = np.maximum(prophet_pred, 0)

prophet_mae  = mean_absolute_error(test_daily, prophet_pred)
prophet_rmse = np.sqrt(mean_squared_error(test_daily, prophet_pred))
prophet_mape = np.mean(np.abs((test_daily.values - prophet_pred) / test_daily.values)) * 100

print(f'✅ Prophet trained!')
print(f'   MAE  = {prophet_mae:.4f} kW')
print(f'   RMSE = {prophet_rmse:.4f} kW')
print(f'   MAPE = {prophet_mape:.2f}%')

In [ ]:
# ── Prophet Component Plots ───────────────────────────────────────────────────
print('📊 Plotting Prophet components...')
fig_comp = prophet_model.plot_components(prophet_forecast)
fig_comp.suptitle('Prophet Model — Decomposed Components',
                  fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('task3_prophet_components.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Prophet component plots saved!')

## 🤖 Step 8: XGBoost Model

In [ ]:
# ── XGBoost with engineered features ─────────────────────────────────────────
print('🔄 Training XGBoost model...')

# Use daily features for XGBoost
df_daily_feat = create_time_features(df_sub.resample('D').mean())
df_daily_feat.dropna(inplace=True)

feat_cols_xgb = [c for c in df_daily_feat.columns if c != 'Global_active_power']
SPLIT = -len(test_daily)
X_tr_xgb = df_daily_feat[feat_cols_xgb].iloc[:SPLIT]
y_tr_xgb = df_daily_feat['Global_active_power'].iloc[:SPLIT]
X_te_xgb = df_daily_feat[feat_cols_xgb].iloc[SPLIT:]
y_te_xgb = df_daily_feat['Global_active_power'].iloc[SPLIT:]

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30
)
xgb_model.fit(
    X_tr_xgb, y_tr_xgb,
    eval_set=[(X_te_xgb, y_te_xgb)],
    verbose=False
)

xgb_pred = np.maximum(xgb_model.predict(X_te_xgb), 0)

xgb_mae  = mean_absolute_error(y_te_xgb, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_te_xgb, xgb_pred))
xgb_mape = np.mean(np.abs((y_te_xgb.values - xgb_pred) / y_te_xgb.values)) * 100

print(f'✅ XGBoost trained (best iteration: {xgb_model.best_iteration})')
print(f'   MAE  = {xgb_mae:.4f} kW')
print(f'   RMSE = {xgb_rmse:.4f} kW')
print(f'   MAPE = {xgb_mape:.2f}%')

## 📊 Step 9: Model Comparison & Visualization

In [ ]:
# ── Figure 4: Actual vs Forecasted (All Models) ───────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(20, 20))
fig.suptitle('Energy Consumption Forecasting — Actual vs Predicted',
             fontsize=17, fontweight='bold', color='#ccccee', y=1.01)

test_index = test_daily.index
all_preds  = {'ARIMA': arima_pred, 'Prophet': prophet_pred, 'XGBoost': xgb_pred}
model_colors = {'ARIMA': ARIMA_COLOR, 'Prophet': PROPHET_COLOR, 'XGBoost': XGBOOST_COLOR}
metrics_all = {
    'ARIMA'  : (arima_mae, arima_rmse, arima_mape),
    'Prophet': (prophet_mae, prophet_rmse, prophet_mape),
    'XGBoost': (xgb_mae, xgb_rmse, xgb_mape)
}

# All models on one plot
axes[0].plot(test_daily.index, test_daily.values, 'o-',
             color=ACTUAL_COLOR, lw=2.5, ms=5, label='Actual', zorder=5)
for mname, mpred in all_preds.items():
    axes[0].plot(test_index[:len(mpred)], mpred, '--',
                 color=model_colors[mname], lw=2, label=mname, alpha=0.9)
axes[0].set_title('All Models — Actual vs Forecast', fontweight='bold', pad=10)
axes[0].set_ylabel('Power (kW)'); axes[0].legend(fontsize=11); axes[0].grid(True, alpha=0.3)

# Individual model subplots
for ax, (mname, mpred) in zip(axes[1:], all_preds.items()):
    mae, rmse, mape = metrics_all[mname]
    color = model_colors[mname]
    idx = test_index[:len(mpred)]
    act = test_daily.values[:len(mpred)]
    residuals = act - mpred
    ax.plot(idx, act, 'o-', color=ACTUAL_COLOR, lw=2, ms=4, label='Actual')
    ax.plot(idx, mpred, 's--', color=color, lw=2, ms=4,
            label=f'{mname} | MAE={mae:.3f} | RMSE={rmse:.3f} | MAPE={mape:.1f}%')
    ax.fill_between(idx, act, mpred, alpha=0.15, color=color)
    ax.set_title(f'{mname} — Actual vs Predicted', fontweight='bold', pad=10)
    ax.set_ylabel('Power (kW)'); ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task3_forecasts.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Forecast plots saved!')

In [ ]:
# ── Figure 5: Model Comparison & XGBoost Feature Importance ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Model Evaluation Summary',
             fontsize=16, fontweight='bold', color='#ccccee', y=1.02)

model_names = ['ARIMA', 'Prophet', 'XGBoost']
mae_vals    = [arima_mae, prophet_mae, xgb_mae]
rmse_vals   = [arima_rmse, prophet_rmse, xgb_rmse]
mape_vals   = [arima_mape, prophet_mape, xgb_mape]
bar_colors  = [ARIMA_COLOR, PROPHET_COLOR, XGBOOST_COLOR]

# MAE comparison
bars = axes[0].bar(model_names, mae_vals, color=bar_colors, edgecolor='white', linewidth=1)
for bar, v in zip(bars, mae_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                 f'{v:.4f}', ha='center', fontsize=12, color='white', fontweight='bold')
axes[0].set_title('Mean Absolute Error (MAE)\n⬇️ Lower = Better', fontweight='bold', pad=12)
axes[0].set_ylabel('MAE (kW)')

# RMSE comparison
bars2 = axes[1].bar(model_names, rmse_vals, color=bar_colors, edgecolor='white', linewidth=1)
for bar, v in zip(bars2, rmse_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                 f'{v:.4f}', ha='center', fontsize=12, color='white', fontweight='bold')
axes[1].set_title('Root Mean Squared Error (RMSE)\n⬇️ Lower = Better', fontweight='bold', pad=12)
axes[1].set_ylabel('RMSE (kW)')

# XGBoost Feature Importance
fi_series = pd.Series(xgb_model.feature_importances_, index=feat_cols_xgb).nlargest(15).sort_values()
fi_colors = plt.cm.plasma(np.linspace(0.3, 0.9, 15))
axes[2].barh(fi_series.index, fi_series.values, color=fi_colors, edgecolor='none')
axes[2].set_title('XGBoost Feature Importance\n(Top 15)', fontweight='bold', pad=12)
axes[2].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('task3_model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

# ── Summary Table ─────────────────────────────────────────────────────────────
print('='*65)
print('  FINAL MODEL PERFORMANCE COMPARISON')
print('='*65)
summary = pd.DataFrame({
    'Model': model_names,
    'MAE (kW)':  [f'{v:.4f}' for v in mae_vals],
    'RMSE (kW)': [f'{v:.4f}' for v in rmse_vals],
    'MAPE (%)':  [f'{v:.2f}' for v in mape_vals]
})
summary.set_index('Model', inplace=True)
print(summary.to_string())

best_model = model_names[np.argmin(mae_vals)]
print(f'\n🏆 Best Model (lowest MAE): {best_model}')
print('\n🎉 Task 3 Complete — Energy Forecasting Done!')